In [ ]:
import numpy as np
import pandas as pd
import skimage as ski
import matplotlib.pyplot as plt
import tifffile
import zarr
import os
import xml.etree.ElementTree as ET
import napari
from importlib import reload
from instanseg import InstanSeg
import load_assess_image as load_assess
import qc_widget
from skimage.measure import regionprops_table
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))  # shows your GPU name


In [11]:
root_dir=rf'D:\thu\HTAN\images'
main_dir = rf'D:\thu\HTAN\images\NST'

In [13]:
save_dir=rf"{root_dir}\mask\test\ILC\qc"
os.makedirs(save_dir,exist_ok=True)

In [ ]:
file_name= 'TNP-TMA-2-Scene-C09.ome.tif'
image_test = load_assess.AssessImage(main_dir, file_name)

In [18]:
def process_MIP(main_dir, filename, cytomarker:list=['CD45','aSMA','panCK','Vim','PDPN','CK7']) -> tuple[str, np.ndarray, np.ndarray]:
    image = load_assess.AssessImage(main_dir, filename)
    image_full = zarr.open(image.loader.image_store, mode='r')
    dna_idx = 0
    dna = np.array(image_full['0'][dna_idx])
    cyto = image.max_projection(view_list_2, resolution_level=0)

    image.loader.tif.close()
    image.loader.image_store.close()
    return filename, dna, cyto


In [19]:
def get_qc_warning(df, round_dna):
    match = df.loc[df['round_dna'] == f'dna({round_dna})', 'warning']
    if match.empty:
        return None
    return match.iloc[0]

# Segmentation and Quantification for single file

In [ ]:
view_list_2=['CD45','aSMA','panCK','Vim','PDPN','CK7']

In [26]:
file_name, dna, cyto = process_MIP(main_dir,
                                    filename =file_name,
                                    cytomarker=view_list_2)

In [31]:
"""
# TENSOR
* labeled_output is a tuple. InstanSeg returns (segmentation_tensor, ...). So labeled_output[0] gets the segmentation tensor.
* The tensor shape is (batch, channels, height, width):
labeled_output[0].shape → (1, 2, H, W)
                           │  │
                           │  └── 2 channels: 0=nuclei, 1=whole cell
                           └───── batch size = 1 (one image)
* labeled_output[0][0, 0]   # batch 0, channel 0 → nuclei mask  (H, W)
* labeled_output[0][0, 1]   # batch 0, channel 1 → cells mask   (H, W)

* .cpu()       # move tensor from GPU memory → CPU memory (required before numpy)
* .numpy()     # convert PyTorch tensor → numpy array
*.astype(int) # each pixel value = instance ID (0=background, *1=cell1, 2=cell2...)

"""

'\n# TENSOR\n* labeled_output is a tuple. InstanSeg returns (segmentation_tensor, ...). So labeled_output[0] gets the segmentation tensor.\n* The tensor shape is (batch, channels, height, width):\nlabeled_output[0].shape → (1, 2, H, W)\n                           │  │\n                           │  └── 2 channels: 0=nuclei, 1=whole cell\n                           └───── batch size = 1 (one image)\n* labeled_output[0][0, 0]   # batch 0, channel 0 → nuclei mask  (H, W)\n* labeled_output[0][0, 1]   # batch 0, channel 1 → cells mask   (H, W)\n\n* .cpu()       # move tensor from GPU memory → CPU memory (required before numpy)\n* .numpy()     # convert PyTorch tensor → numpy array\n*.astype(int) # each pixel value = instance ID (0=background, *1=cell1, 2=cell2...)\n\n'

In [28]:


model = InstanSeg('fluorescence_nuclei_and_cells', device='cuda')

for element in image_test.loader.root.iter():
    if element.tag.endswith('Pixels'):
        pixel_size = float(element.attrib['PhysicalSizeX'])

stack = np.stack([dna, cyto])
labeled_output = model.eval_medium_image(stack, pixel_size=pixel_size)

# Save mask in NPZ form - faster to reload
stem = file_name.replace('.ome.tif', '')   # → 'TNP-TMA-29_A7-ILC-3A'
output_path = os.path.join(save_dir, stem + '.npz')
nuclei_mask = labeled_output[0][0, 0].cpu().numpy().astype(int)
cells_mask  = labeled_output[0][0, 1].cpu().numpy().astype(int)
# np.savez_compressed(output_path, nuclei=nuclei_mask, cells=cells_mask)

Model fluorescence_nuclei_and_cells version 0.1.1 already downloaded in d:\conda_envs\instanseg\Lib\site-packages\instanseg\utils\../bioimageio_models/, loading


d:\conda_envs\instanseg\Lib\site-packages\instanseg\utils\pytorch_utils.py:312: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\SparseCsrTensorImpl.cpp:55.)
  intersection = torch.sparse.mm(onehot1, onehot2.T).to_dense()


# View mask

In [30]:
image_test=load_assess.AssessImage(main_dir,file_name)
viewer = image_test.view_images(biomarkers_of_interest=['dna'], resolution_level=0)
viewer.add_image(cyto, name='MIP',blending='additive', colormap='magenta')
viewer.add_labels(nuclei_mask, name='nuclei')
viewer.add_labels(cells_mask,  name='cells')

Finding biomarkers: 100%|██████████| 54/54 [00:00<00:00, 57.17it/s]


<Labels layer 'cells' at 0x20c579a5cd0>

# Quantification of nuclei and cells

In [ ]:
# shape props once — independent of biomarker
shape_props = regionprops_table(
    cells_mask,
    properties=['label', 'area', 'centroid', 'equivalent_diameter_area',
                'solidity', 'eccentricity', 'axis_major_length', 'axis_minor_length', 'perimeter']
)
df = pd.DataFrame(shape_props)

# intensity per biomarker — rename columns to avoid collision, merge on label
for biomarker, channel_index in image_test.biomarker_channel.items():
    image_biomarker = np.array(open_image_test[str(resolution_level)][int(channel_index)])
    intensity_props = regionprops_table(
        cells_mask,
        intensity_image=image_biomarker,
        properties=['label', 'intensity_mean', 'intensity_median', 'intensity_std']
    )
    intensity_df = pd.DataFrame(intensity_props).rename(columns={
        'intensity_mean':   f'{biomarker}_mean',
        'intensity_median': f'{biomarker}_median',
        'intensity_std':    f'{biomarker}_std',
    })
    df = df.merge(intensity_df, on='label')

output_path = os.path.join(r'r:\Thu\From_2026\6. Other\DL_Summer_2026\output',
                           file_name.replace('.ome.tif', '_cells.csv'))
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} cells → {output_path}")
